# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [230]:
# Write your code below.
%load_ext dotenv
%dotenv



The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [231]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [ ]:
import os
from glob import glob

# Write your code below.

# price_data_location 
PRICE_DATA_location = os.getenv("PRICE_DATA")

# parquet_files_location 
parquet_files = glob(os.path.join(PRICE_DATA_location, "**/*.parquet"), recursive = True)



In [ ]:
# debug only
#PRICE_DATA
#parquet_files

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [ ]:
# Write your code below.

# read all parquet files
dd_px = dd.read_parquet(parquet_files)

# Ensure proper sorting within each ticker 
dd_px = dd_px.map_partitions(lambda df: df.sort_values(["ticker", "Date"]))

# debug only
#dd_px

# create dd_shift function
dd_shift = (
    dd_px
        .groupby('ticker', group_keys=False)
        .apply(
            lambda x: x.sort_values('Date', ascending = True)
                       .assign(
                           Close_lag_1 = x['Close'].shift(1),
                           dd_feat = x['High']-x['Low']
                )
))

# debug only
#dd_shift


dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1
   )



In [246]:
# debug only
dd_rets

,Date,Open,High,Low,Close,Adj Close,Volume,source,ticker,Year,Close_lag_1,dd_feat,Returns
npartitions=2797,,,,,,,,,,,,,
,datetime64[ns],float64,float64,float64,float64,float64,float64,string,string,int32,float64,float64,float64
,...,...,...,...,...,...,...,...,...,...,...,...,...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...
,...,...,...,...,...,...,...,...,...,...,...,...,...


+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [ ]:
# Write your code below.
# Convert to pandas data frame. 
pd_px = dd_rets.compute()

# Sort to proper time order
pd_px = pd_px.sort_values(['ticker', 'Date'])

# 10-day moving average of returns for each ticker
pd_px['returns_mean_10day'] = (
    pd_px.groupby('ticker')['Returns']
       .transform(lambda x: x.rolling(10).mean())
)


In [ ]:
# debug only
pd_px.info()

<class 'pandas.core.frame.DataFrame'>
Index: 324342 entries, 137315 to 88147
Data columns (total 14 columns):
 #   Column              Non-Null Count   Dtype         
---  ------              --------------   -----         
 0   Date                324342 non-null  datetime64[ns]
 1   Open                324337 non-null  float64       
 2   High                324337 non-null  float64       
 3   Low                 324337 non-null  float64       
 4   Close               324337 non-null  float64       
 5   Adj Close           324337 non-null  float64       
 6   Volume              324337 non-null  float64       
 7   source              324342 non-null  string        
 8   ticker              324342 non-null  string        
 9   Year                324342 non-null  int32         
 10  Close_lag_1         324247 non-null  float64       
 11  dd_feat             324337 non-null  float64       
 12  Returns             324242 non-null  float64       
 13  returns_mean_10day  322647 non

Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

Converting to pandas is not required, but it can make calculating the moving average simpler.

For large datasets, Dask generally performs better than pandas due to its parallel processing and scalable pipeline design.
 
(1 pt)

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.